# 01 — Validação das fontes externas

Checagens locais (sobre `data/raw/`, sem custo de BigQuery) antes da construção das features:

1. A `taxa_alfabetizacao` do Censo 2022 corresponde à população de 15 anos ou mais?
2. Os tipos de esgotamento sanitário têm sobreposição (categoria agregada)?
3. FUNDEB/SIOPE: em qual bimestre está o consolidado anual e qual coluna de valor cada indicador usa?
4. Cobertura de cada fonte sobre os 5.232 municípios da base de modelagem.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

RAW = Path("data/raw")

gold = pd.read_parquet(RAW / "gold_base.parquet")
censo = pd.read_parquet(RAW / "censo2022_municipio.parquet")
alf = pd.read_parquet(RAW / "censo2022_alfabetizacao.parquet")
esg = pd.read_parquet(RAW / "censo2022_esgotamento.parquet")
fundeb = pd.read_parquet(RAW / "fundeb_2023.parquet")
alunos = pd.read_parquet(RAW / "silver_alunos.parquet")
evol = pd.read_parquet(RAW / "gold_evolucao.parquet")

ids_gold = set(gold["id_municipio"].astype(str))
print("Municípios na base de modelagem:", len(ids_gold))

Municípios na base de modelagem: 5232


## 1. Taxa de alfabetização do Censo 2022 = população de 15 anos ou mais?

A tabela detalhada só contém faixas a partir de 15 anos. Se a taxa recalculada a partir dela coincidir com a `taxa_alfabetizacao` da tabela `municipio`, a definição está confirmada.

In [2]:
contagem = alf.pivot_table(
    index="id_municipio", columns="alfabetizacao", values="populacao", aggfunc="sum"
)
taxa_15_calc = 100 * contagem["Alfabetizadas"] / contagem.sum(axis=1)

comp = (
    censo.set_index("id_municipio")[["taxa_alfabetizacao"]]
    .join(taxa_15_calc.rename("taxa_15_calc"))
)
# A tabela `municipio` armazena a taxa na escala 0–1; a recalculada está em 0–100.
comp["taxa_oficial_pct"] = 100 * comp["taxa_alfabetizacao"]
comp["diferenca"] = comp["taxa_oficial_pct"] - comp["taxa_15_calc"]

print(comp.head().to_string())
print("\nDiferença (oficial em % - recalculada), mesma escala:")
print(comp["diferenca"].describe().to_string())

              taxa_alfabetizacao  taxa_15_calc  taxa_oficial_pct  diferenca
id_municipio                                                               
1400704                  0.87529      87.52867            87.529    0.00033
1200435                  0.77397     77.396543            77.397   0.000457
1400407                  0.91823     91.823247            91.823  -0.000247
1200328                  0.81876     81.875698            81.876   0.000302
1303908                  0.85195     85.194818            85.195   0.000182

Diferença (oficial em % - recalculada), mesma escala:
count      5570.0
mean    -0.000006
std       0.00029
min     -0.000499
25%     -0.000259
50%     -0.000009
75%      0.000243
max        0.0005


## 2. Esgotamento sanitário: hierarquia entre os tipos?

Hipótese: `Rede geral, rede pluvial ou fossa ligada à rede` = `Rede geral ou pluvial` + `Fossa séptica ou fossa filtro ligada à rede`. Se confirmada, somar os 9 tipos conta parte da população duas vezes.

In [3]:
esg_w = esg.pivot_table(
    index="id_municipio", columns="tipo_esgotamento_sanitario", values="populacao", aggfunc="sum"
).fillna(0)

AGREGADO = "Rede geral, rede pluvial ou fossa ligada à rede"
COMPONENTES = ["Rede geral ou pluvial", "Fossa séptica ou fossa filtro ligada à rede"]

print("Totais nacionais por tipo:")
print(esg_w.sum().sort_values(ascending=False).to_string())

dif_hierarquia = esg_w[AGREGADO] - esg_w[COMPONENTES].sum(axis=1)
print("\nAgregado - soma dos componentes (por município):")
print(dif_hierarquia.describe().to_string())

pop_esg_sem_agregado = esg_w.drop(columns=AGREGADO).sum(axis=1)
razao_pop = pop_esg_sem_agregado / censo.set_index("id_municipio")["populacao"]
print("\nPopulação coberta pela tabela de esgotamento / população total do Censo:")
print(razao_pop.describe().to_string())

Totais nacionais por tipo:
tipo_esgotamento_sanitario
Rede geral, rede pluvial ou fossa ligada à rede    126320738
Rede geral ou pluvial                              117771612
Fossa rudimentar ou buraco                          39290587
Fossa séptica ou fossa filtro não ligada à rede     26726073
Fossa séptica ou fossa filtro ligada à rede          8549126
Rio, lago, córrego ou mar                            4072542
Vala                                                 2980118
Outra forma                                          1503483
Não tinham banheiro nem sanitário                    1182315

Agregado - soma dos componentes (por município):
count    5570.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0

População coberta pela tabela de esgotamento / população total do Censo:
count      5570.0
mean     0.999967
std      0.001929
min      0.872631
25%           1.0
50%           1.0
75%           1.0
max      1.066697


## 3. FUNDEB/SIOPE: bimestre do consolidado anual e coluna de valor por indicador

In [4]:
resumo_fundeb = (
    fundeb.groupby(["id_indicador", "bimestre"])
    .agg(
        municipios=("id_municipio", "nunique"),
        nulos_real=("valor_real", lambda s: s.isna().sum()),
        nulos_pct=("valor_percentual", lambda s: s.isna().sum()),
        mediana_real=("valor_real", "median"),
        mediana_pct=("valor_percentual", "median"),
    )
    .reset_index()
)
print(resumo_fundeb.to_string(index=False))

id_indicador  bimestre  municipios  nulos_real  nulos_pct  mediana_real  mediana_pct
          35         1        5563        5563          0           NaN        11.96
          35         2        5563        5563          0           NaN        19.25
          35         3        5564        5564          0           NaN        22.54
          35         4        5566        5566          0           NaN        24.55
          35         5        5565        5565          0           NaN        25.82
          35         6        5566        5566          0           NaN        23.94
          44         1        5566           0       5566      1150.215          NaN
          44         2        5566           0       5566      2551.665          NaN
          44         3        5566           0       5566      3990.270          NaN
          44         4        5566           0       5566      5459.690          NaN
          44         5        5566           0       5566      70

## 4. Cobertura das fontes sobre a base de modelagem

In [5]:
ids_por_fonte = {
    "censo2022_municipio": set(censo["id_municipio"]),
    "censo2022_alfabetizacao": set(alf["id_municipio"]),
    "censo2022_esgotamento": set(esg["id_municipio"]),
    "fundeb_2023": set(fundeb["id_municipio"]),
    "alunos_2024 (rede 3)": set(alunos.loc[alunos["ano"] == 2024, "id_municipio"].astype(str)),
    "indicador_2023": set(evol["id_municipio"].astype(str)),
}

for nome, ids in ids_por_fonte.items():
    print(f"{nome:<26} {len(ids_gold & ids):>5} de {len(ids_gold)}")

print("\nAlunos avaliados por município (rede municipal, 2024):")
print(alunos.loc[alunos["ano"] == 2024, "qtd_alunos"].describe().to_string())

censo2022_municipio         5232 de 5232
censo2022_alfabetizacao     5232 de 5232
censo2022_esgotamento       5232 de 5232
fundeb_2023                 5230 de 5232
alunos_2024 (rede 3)        5232 de 5232
indicador_2023              4611 de 5232

Alunos avaliados por município (rede municipal, 2024):
count         5452.0
mean      337.541636
std      1286.749357
min             10.0
25%             55.0
50%            114.0
75%            256.0
max          49820.0


## Conclusões

1. **Alfabetização (Censo 2022):** a `taxa_alfabetizacao` da tabela `municipio` está na escala 0–1; multiplicada por 100, coincide com a taxa recalculada a partir da tabela detalhada, que só contém faixas a partir de 15 anos. Definição confirmada: **população de 15 anos ou mais**. A tabela detalhada permite também a taxa de **25 a 44 anos** (geração dos pais das crianças do 2º ano).
2. **Esgotamento sanitário:** hierarquia confirmada — `Rede geral, rede pluvial ou fossa ligada à rede` é exatamente a soma de `Rede geral ou pluvial` e `Fossa séptica ou fossa filtro ligada à rede` (diferença 0 em todos os municípios). Totais devem excluir o agregado; sem ele, a população coberta equivale à do Censo. Features: % com rede/fossa ligada à rede e % com esgotamento precário (fossa rudimentar, vala, rio/lago/mar, outra forma, sem banheiro).
3. **FUNDEB/SIOPE:** valores acumulados ao longo do ano — o **bimestre 6** é o consolidado anual. Indicadores em R$/aluno usam `valor_real`; percentuais, `valor_percentual`. O indicador 61 foi descartado (mediana 0,00 em todos os bimestres).
4. **Cobertura sobre os 5.232 municípios:** Censo 2022 e número de alunos completos; FUNDEB 5.230 (2 faltantes, tratados por imputação no pipeline); `indicador_2023` 4.611 (ausência de SP na fonte).
5. **Alunos avaliados (rede municipal, 2024):** mediana de 114; 25% dos municípios com até 55 alunos. Com 55 alunos e taxa em torno de 63%, o erro-padrão binomial da taxa é de ~6,5 p.p. — a ponderação do treino pelo número de alunos é necessária.